# Session 2 – Apache Spark Fundamentals
**FH Kufstein Tirol · Big Data Processing (DPR.1) · Winter Semester 2026/27**

---

> 💡 **No Java on your laptop?** The same content (plus Spark ML) is in `session-04/04_spark_fundamentals_and_ml.ipynb`, which runs on Kaggle.

## What you will do in this notebook

On the slides you saw Spark's architecture: Driver, Executors, SparkSession, RDDs, DataFrames, and the Catalyst Optimizer. Today you will **run** all of it.

By the end you will be able to:
- Create a SparkSession and explain what it does.
- Distinguish **transformations** from **actions** and predict when computation starts.
- Read and explore Parquet data with DataFrames.
- Apply `select`, `filter`, `withColumn`, `groupBy`, `agg`, `orderBy`.
- Read a physical execution plan and identify where a **shuffle** occurs.
- Answer exam-style questions about lazy evaluation and partitioning.

---

## ⚡ Pre-flight checklist (do this first, before anything else)

1. **Close all other notebook kernels** – VS Code: *Jupyter* panel → right-click each other kernel → *Shut Down Kernel*.
2. **Check available memory** – run the cell below (works on Linux, macOS and Windows). You need ≥ 4.5 GB available before starting Spark.
3. **Check your data** – run the second cell. You need at least one month of Parquet data.
4. **Work with one month only today** – this keeps memory usage well below the 8 GB Codespace limit.

In [ ]:
# Pre-flight: memory check (psutil works on Linux, macOS and Windows)
import psutil
vm = psutil.virtual_memory()
print(f"RAM total     : {vm.total/1e9:5.1f} GB")
print(f"RAM available : {vm.available/1e9:5.1f} GB")
print("✅ enough memory for Spark" if vm.available/1e9 >= 4.5
      else "⚠️ less than 4.5 GB available – close other notebooks and browser tabs first")

In [ ]:
# Pre-flight: data check
import glob, os

parquet_files = sorted(glob.glob("../data/yellow_tripdata_2024-*.parquet"))
zone_file     = "../data/taxi_zone_lookup.csv"

print(f"Parquet files found: {len(parquet_files)}")
for f in parquet_files:
    size_mb = os.path.getsize(f) / 1_048_576
    print(f"  {os.path.basename(f):45s}  {size_mb:6.1f} MB")

print()
if not parquet_files:
    print("❌ No data found. Run: python get_data.py --months 1")
else:
    print("✅ Parquet data OK")

if os.path.exists(zone_file):
    print("✅ Taxi zone lookup OK")
else:
    print("❌ Taxi zone lookup missing. Run: python get_data.py --months 1")

---

## Part 1 – The SparkSession: your entry point into Spark

### What you already know (from Session 3)

| Component | Role |
|-----------|------|
| **Driver** | Your Python process. Runs your code, plans queries, coordinates execution. |
| **Cluster Manager** | Allocates resources (here: *local* mode – no real cluster). |
| **Executors** | JVM processes that read data and execute tasks. |
| **SparkSession** | The single entry point to all Spark functionality since Spark 2.0. |

In **local mode** (`local[*]`), the Driver and Executors all run inside one JVM on your machine. You get the full API — the only difference from a real cluster is that there is no network between nodes.

### 🔮 Predict before you run

> **Q1 – Before running the cell below:** What does calling `SparkSession.builder...getOrCreate()` actually do? Does it read any data? Does it start any computation?
>
> *Write your answer here (double-click to edit):*
>
> ...

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("SparkFundamentals")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")   # default is 200 – too many for a small Codespace
    .getOrCreate()
)

print(f"Spark version : {spark.version}")
print(f"App name      : {spark.sparkContext.appName}")
print(f"Master        : {spark.sparkContext.master}")
print()
print("Spark UI available at: http://localhost:4040  (open in a browser tab)")

**A look at the Spark UI**

Open `http://localhost:4040` in a new browser tab. You will see:
- **Jobs** – each Action creates one or more jobs.
- **Stages** – each job is divided into stages; stage boundaries are created by shuffles.
- **Tasks** – the smallest unit of work, one per partition per stage.
- **Storage** – cached DataFrames.
- **Environment** – all config values (useful for debugging).

Right now the UI is mostly empty. It will fill up as you run actions below.

---

## Part 2 – Reading data: from Parquet to DataFrame

A **DataFrame** is Spark's primary data abstraction since Spark 2.0. It is:
- Distributed across partitions (files, HDFS blocks, …).
- Typed with a **schema** known to the Driver at planning time.
- Processed through the **Catalyst Optimizer** (logical plan → optimised physical plan).

Unlike an RDD (which is just a distributed collection of objects), a DataFrame lets Catalyst inspect column types and reorder or push down operations automatically.

### 🔮 Predict before you run

> **Q2 –** When you call `spark.read.parquet(...)`, does Spark immediately read all the data into memory?
>
> *Your answer:*
>
> ...

In [ ]:
# Use January 2024 only – enough data, fits comfortably in the Codespace
DATA_PATH = "../data/yellow_tripdata_2024-01.parquet"

df = spark.read.parquet(DATA_PATH)

print("DataFrame created. Schema:")
df.printSchema()

In [ ]:
# How many rows are there?
# Note: this IS an Action – it triggers computation.
row_count = df.count()
print(f"Total rows in January 2024: {row_count:,}")

In [ ]:
# Look at a few rows
# show() is also an Action.
df.show(5, truncate=False)

In [ ]:
# Basic statistics for numeric columns
df.select(
    "trip_distance", "fare_amount", "passenger_count"
).describe().show()

---

## Part 3 – Transformations vs. Actions: the most important concept in Spark

> **Rule:** Spark is **lazy**. A *transformation* describes what to do but does **not** start execution. Only an *action* sends the job to the Executors.

| | Transformations | Actions |
|---|---|---|
| **What they do** | Build an execution plan | Trigger execution, return a result |
| **Return value** | A new DataFrame | A Python value or side-effect |
| **Examples** | `select` `filter` `withColumn` `groupBy` `join` `orderBy` | `count` `show` `collect` `take` `write` |
| **Cost** | Nearly free (just graph nodes) | Executes the full plan |

### 🔮 Predict before you run

> **Q3 –** Read the four lines below **before** running the cell. Which lines start a Spark job? Which ones are free?
>
> ```python
> t1 = df.filter(F.col("trip_distance") > 1.0)           # line A
> t2 = t1.select("PULocationID", "trip_distance")         # line B
> t3 = t2.groupBy("PULocationID").count()                  # line C
> result = t3.show(10)                                     # line D
> ```
>
> *Your answer (A / B / C / D – which triggers computation?):*
>
> ...

In [ ]:
# Now run it and check the Spark UI after each line
t1 = df.filter(F.col("trip_distance") > 1.0)          # transformation
print("t1 defined – no job yet")

t2 = t1.select("PULocationID", "trip_distance")        # transformation
print("t2 defined – still no job")

t3 = t2.groupBy("PULocationID").count()                # transformation (groupBy + count = wide)
print("t3 defined – still no job")

print("\nNow triggering an action:")
t3.show(10)                                             # ACTION – job runs now

**Check the Spark UI now.** You should see one completed job. Click on it — you will see how many stages it has and how many tasks ran.

---

### Chaining transformations

In practice, transformations are chained with the dot notation. Spark builds one combined logical plan and optimises it.

In [ ]:
# Same logic, written as a chain – identical result, identical plan
(
    df
    .filter(F.col("trip_distance") > 1.0)
    .select("PULocationID", "trip_distance")
    .groupBy("PULocationID")
    .count()
    .show(10)
)

---

## Part 4 – Core DataFrame Operations

Work through these cells one by one. After each cell, note what the operation does and whether it is a transformation or an action.

In [ ]:
# --- 4.1  select: choose and rename columns ---
(
    df.select(
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        F.col("trip_distance").alias("distance_miles"),
        "fare_amount",
        "PULocationID",
        "DOLocationID"
    )
    .show(5)
)

In [ ]:
# --- 4.2  filter: keep only valid trips ---
# Bad rows: negative fares, zero distance, zero or null passengers
clean = (
    df
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("passenger_count") >= 1)
)

print(f"Original rows : {df.count():>10,}")
print(f"After cleaning: {clean.count():>10,}")

In [ ]:
# --- 4.3  withColumn: create or replace a column ---
# Add a column with the fare per mile
enriched = clean.withColumn(
    "fare_per_mile",
    F.round(F.col("fare_amount") / F.col("trip_distance"), 2)
)

enriched.select("trip_distance", "fare_amount", "fare_per_mile").show(5)

In [ ]:
# --- 4.4  groupBy + agg: aggregate ---
# Count trips and compute average fare per pickup location
by_location = (
    clean
    .groupBy("PULocationID")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance")
    )
)

by_location.show(10)

In [ ]:
# --- 4.5  orderBy: sort the result ---
(
    by_location
    .orderBy(F.col("trip_count").desc())
    .show(15)
)

---

## Part 5 – Reading the Execution Plan

Every DataFrame has a **plan**. `df.explain()` shows you how Spark intends to execute a query.

```
Logical Plan      →  Catalyst optimises  →  Physical Plan  →  Execution
(what you wrote)                            (how Spark runs it)
```

Key symbols to look for in the physical plan:

| Symbol | Meaning |
|--------|---------|
| `FileScan parquet` | Reading data from Parquet files |
| `Filter` | A predicate applied at scan time (predicate pushdown) |
| `Exchange` | A **shuffle** – data is redistributed across partitions |
| `HashAggregate` | Partial or final aggregation |
| `Sort` | Sorting within a partition |
| `BroadcastHashJoin` | Small table broadcast to all executors (no shuffle) |
| `SortMergeJoin` | Large-large join with shuffle and sort |

### 🔮 Predict before you run

> **Q4 –** Look at the `by_location` DataFrame above (filter → groupBy → agg). 
> Do you expect to see an `Exchange` node in the plan? Why or why not?
>
> *Your answer:*
>
> ...

In [ ]:
# Compact plan (default)
by_location.explain()

In [ ]:
# Extended plan: shows logical plan stages AND the physical plan
by_location.explain(mode="formatted")

**Read the plan together:**

- `FileScan parquet` at the bottom — this is always where Spark starts (bottom to top).
- `Filter` right after the scan — Catalyst pushed the `fare_amount > 0` filter down to the scan.
- `HashAggregate` appears **twice**: once for a partial aggregation (per partition, before shuffle) and once for a final merge (after shuffle). This is a standard Spark optimisation.
- `Exchange` in between — this is the **shuffle**. After the partial aggregation, all rows with the same `PULocationID` must land on the same partition before the final merge can happen.

### Narrow vs. Wide Transformations

| Type | Definition | Examples | Shuffle? |
|------|-----------|---------|----------|
| **Narrow** | Each output partition depends on exactly one input partition | `filter`, `select`, `withColumn`, `map` | No |
| **Wide** | Each output partition depends on multiple input partitions | `groupBy`, `join`, `orderBy`, `repartition` | **Yes** |

Stage boundaries in Spark are always at wide transformations.

In [ ]:
# Contrast: a filter-only plan has NO Exchange
filter_only = clean.filter(F.col("trip_distance") > 10)
filter_only.explain()

---

## Part 6 – Group Work: NYC Taxi Analysis 🚕

Work in groups of **3–4 people**. Rotate roles every 20 minutes:

| Role | Responsibility |
|------|----------------|
| **Driver** | Types and runs the code |
| **Navigator** | Reads the task, checks logic, predicts results |
| **Explainer** | Writes the explanation in Markdown cells |
| **Reviewer** (4 people) | Challenges assumptions, checks edge cases |

**Rule:** All group members must be able to explain every line. The exam is individual and without AI.

---

### Task A – Core (everyone)

**Question:** *Which pickup location IDs generated the most taxi trips in January 2024?*

Your pipeline must:
1. Start from the cleaned DataFrame (`clean`).
2. Aggregate to get trip count and average fare per pickup location.
3. Sort descending by trip count.
4. Show the top 15 results.
5. In the Markdown cell below the code, explain: *where does the shuffle happen and why?*

In [ ]:
# Task A – your solution here

top_locations = (
    # YOUR CODE
)

top_locations.show(15)

**Task A – Explanation (Explainer fills this in):**

- The shuffle happens at: *...*
- It is necessary because: *...*
- The `Exchange` node in the plan shows: *...*

---

### Task B – Extension: add location names (if Task A is done)

The result of Task A shows only numeric location IDs. Join the taxi zone lookup table to add zone names.

**Steps:**
1. Read `../data/taxi_zone_lookup.csv` into a DataFrame called `zones`.
2. Inspect the schema — which column is the join key?
3. Join `top_locations` with `zones` to add `Borough` and `Zone`.
4. Show the top 15 with location name, borough, trip count, and average fare.
5. Run `explain()` on the joined DataFrame. What kind of join does Spark choose? Why?

In [ ]:
# Task B – read the zones table
zones = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("../data/taxi_zone_lookup.csv")
)

zones.printSchema()
zones.show(5)

In [ ]:
# Task B – join and show
# Hint: the zone table is small (~265 rows). What join strategy do you expect Spark to pick?

result_with_names = (
    # YOUR CODE
)

result_with_names.show(15, truncate=False)

In [ ]:
# Task B – inspect the join plan
result_with_names.explain(mode="formatted")

**Task B – Explanation:**

- Spark chose a **[BroadcastHashJoin / SortMergeJoin]** because: *...*
- A BroadcastHashJoin avoids a shuffle because: *...*
- The threshold for automatic broadcasting is controlled by: *...*

---

### Task C – Plan Challenge (for fast groups)

Add an `orderBy` to the joined result and run `explain()` again.

1. Does adding a sort introduce a new `Exchange`? Why or why not?
2. Where in the plan does the sort appear relative to the join?
3. Open the Spark UI (`http://localhost:4040`): how many stages does this job have? Why?

In [ ]:
# Task C – your solution and explain()

# YOUR CODE

**Task C – Explanation:**

*...*

---

## Part 7 – A Note on `collect()` and `toPandas()`

Two actions that are **dangerous on large DataFrames**:

In [ ]:
# SAFE: collect on a small, already-aggregated result
top5 = by_location.orderBy(F.col("trip_count").desc()).limit(5)
rows = top5.collect()   # returns a list of Row objects in the Driver
print(type(rows), len(rows))
for r in rows:
    print(r["PULocationID"], r["trip_count"])

In [ ]:
# DANGEROUS: never do this on the raw, unaggregated DataFrame
# df.collect()         # would pull ~3 million rows into Driver memory → OOM risk
# df.toPandas()        # same problem, plus PyArrow conversion overhead

# SAFE: filter first, then convert a small result to pandas for plotting
top_df = (
    by_location
    .orderBy(F.col("trip_count").desc())
    .limit(20)
    .toPandas()         # small result → fine
)

top_df.head()

In [ ]:
# Quick bar chart of the top 20 pickup locations
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(top_df["PULocationID"].astype(str), top_df["trip_count"])
ax.set_xlabel("PULocationID")
ax.set_ylabel("Trip count")
ax.set_title("Top 20 Pickup Locations – NYC Yellow Taxi, January 2024")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

---

## Part 8 – Housekeeping: stopping Spark

Always stop the SparkSession when you are done. This releases the JVM memory and avoids port conflicts if you restart the notebook.

In [ ]:
# Run this cell when you are finished
spark.stop()
print("SparkSession stopped. Memory released.")

---

## 📝 Exit Ticket – Individual, no AI

Answer the following questions **in your own words**, in this notebook. You have 10 minutes. These are representative of exam questions.

---

**Question 1**

A colleague writes the following code:

```python
df2 = df.filter(F.col("trip_distance") > 5)
df3 = df2.select("PULocationID", "trip_distance")
df4 = df3.groupBy("PULocationID").count()
```

She says: *"Three lines of code, so Spark ran three operations immediately."*  
Is she correct? Explain what actually happens and when computation starts.

*Your answer:*

> ...

---

**Question 2**

You run a `groupBy("PULocationID").count()` on a DataFrame with 8 partitions. Explain:
- Why does this require a shuffle?
- What does Spark do in the two `HashAggregate` steps you see in the plan?

*Your answer:*

> ...

---

**Question 3**

The following query returns exactly 10 rows:

```python
df.filter(F.col("fare_amount") > 50).show(10)
```

Does this mean Spark only read 10 rows from the Parquet file? Explain the actual behaviour and why it matters for performance.

*Your answer:*

> ...

---

**Question 4 (harder)**

You join the trip data (3 million rows) with the taxi zone lookup table (265 rows). The plan shows a `BroadcastHashJoin` instead of a `SortMergeJoin`.
- Why did Spark choose BroadcastHashJoin?
- What is the benefit compared to SortMergeJoin?
- What could go wrong if you accidentally broadcast a very large table?

*Your answer:*

> ...